# Scrapping  Twitter Pemilu 2024 using tweet-harvers


## Data Gathers Twitter

In [ ]:
!pip uninstall nltk

In [ ]:
%pip install nltk

In [ ]:
# # Install Node.js (because tweet-harvest built using Node.js)
# !sudo apt-get update
# !sudo apt-get install -y ca-certificates curl gnupg
# !sudo mkdir -p /etc/apt/keyrings
# !curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | sudo gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg

# !NODE_MAJOR=20 && echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_$NODE_MAJOR.x nodistro main" | sudo tee /etc/apt/sources.list.d/nodesource.list

# !sudo apt-get update
# !sudo apt-get install nodejs -y

# !node -v

In [ ]:
# # Crawl Data

# filename = 'pemilu.csv'
# search_keyword = '#DebatCapres until:2024-01-08 since:2024-01-06'
# limit = 3000

# !npx --yes tweet-harvest@2.2.8 -o "{filename}" -s "{search_keyword}" -l {limit} --token ""

## Import Library

In [ ]:
import pandas as pd
import nltk, re, string

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn import naive_bayes

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn

from nlp_id.lemmatizer import Lemmatizer

from langdetect import detect

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

## Load Dataset

In [ ]:
data = pd.read_csv('DataTweetFix.csv')

display(data)

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

## Data Cleansing

In [ ]:
data = data[['created_at', 'username', 'full_text']]

data.head(10)

In [ ]:
def detect_language(text):
    try:
        lang = detect(text)
        return lang
    except:
        return 'unknown'

data['Language'] = data['full_text'].apply(detect_language)

data = data[data['Language'] == 'id']

In [ ]:
data.head(10)

## Slang Word Changes

In [ ]:
df_slang = pd.read_csv('kamus-alay\colloquial-indonesian-lexicon.csv') 

def replace_slang(text):
    words = text.split()
    for i in range(len(words)):
        slang_word = df_slang[df_slang['slang'] == words[i]]
        if not slang_word.empty:
            words[i] = slang_word['formal'].values[0]
    return ' '.join(words)

In [ ]:
data['full_text'] = data['full_text'].apply(replace_slang)

In [ ]:
data.head(10)

## Data Pre-Processing

In [ ]:
def remove_tweet_special(text):
    # Menghapus tab, baris baru, dan back slice
    text = text.replace('\\t', ' ').replace('\\n', ' ').replace('\\u', ' ').replace('\\', '')

    # Menghapus karakter non-ASCII (emotikon, huruf Cina, dll.)
    text = text.encode('ascii', 'replace').decode('ascii')

    # Menghapus mention, link, dan hashtag
    text = ' '.join(re.sub("([@#][A-Za-z0-9]+)|(\w+:\/\/\S+)", " ", text).split())

    # Menghapus URL yang tidak lengkap
    text = text.replace("http://", " ").replace("https://", " ")

    return text

def clean_text(data):
    # Menghapus karakter yang tidak diinginkan (_) dan karakter selain huruf
    data['full_text'] = data['full_text'].str.replace(r"[^a-zA-Z ]+", " ").str.strip()

    # Mengganti multiple whitespace dengan single whitespace
    data['full_text'] = data['full_text'].replace(" +", " ", regex=True)

    return data

data['full_text'] = data['full_text'].apply(remove_tweet_special)

data = clean_text(data)


In [ ]:
def clean_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

lemmatizer = Lemmatizer()

def lemmatize_text(text):
    return lemmatizer.lemmatize(text)

def remove_numbers(text):
    return re.sub('[0-9]+', '', text)

def remove_stopwords(text):
    stop_words = set(stopwords.words('indonesian'))
    return ' '.join([word for word in nltk.word_tokenize(text) if word.lower() not in stop_words])

data['full_text'] = data['full_text'].apply(lambda x: x.lower())

data['full_text'] = data['full_text'].apply(clean_punctuation)

data['full_text'] = data['full_text'].apply(lemmatize_text)

data['full_text'] = data['full_text'].apply(remove_numbers)

data['full_text'] = data['full_text'].apply(remove_stopwords)

In [ ]:
data.head(10)

## Negative and Positive Word

In [ ]:
file_negative = 'InSet/negative.tsv'
file_positive = 'Inset/positive.tsv'

data_negative = pd.read_csv(file_negative, sep='\t')
data_positive = pd.read_csv(file_positive, sep='\t')

# Tambahkan kolom label sentimen
data_negative['sentiment'] = 'negative'
data_positive['sentiment'] = 'positive'

# Gabungkan dataset negative dan positive menjadi satu dataset
data_combined = pd.concat([data_negative, data_positive], ignore_index=True)

# Tampilkan beberapa contoh data
data_combined.head(10)


In [ ]:
word_weight_dict = dict(zip(data_combined['word'], data_combined['weight']))

def determine_sentiment(word):
    if word in word_weight_dict:
        if word_weight_dict[word] < 0:
            return 'negatif'
        elif word_weight_dict[word] > 0:
            return 'positif'
    return 'netral'

In [ ]:
data.head(10)

## Labeling data

In [ ]:
def test_sentiment(data):
    data['sentiment'] = data['full_text'].apply(lambda x: [determine_sentiment(word) for word in x.split()])
    return data

def calculate_text_weight(text):
    total_weight = 0
    words = text.split()
    for word in words:
        if word in word_weight_dict:
            total_weight += word_weight_dict[word]
    return total_weight

data['text_weight'] = data['full_text'].apply(calculate_text_weight)

data = test_sentiment(data)

data.head(10)

In [ ]:
data['class_sentiment'] = data['text_weight'].apply(lambda x: 'negatif' if x < 0 else ('positif' if x > 0 else 'netral'))

In [ ]:
data['text_weight_label'] = data['text_weight'].apply(lambda x: -1 if x < 0 else (0 if x == 0 else 1))

In [ ]:
data = data[['created_at', 'full_text', 'text_weight', 'text_weight_label', 'class_sentiment']]

data.head(10)


In [ ]:
data.to_csv('data.csv', index=False)

In [ ]:
data['class_sentiment'].value_counts().plot(kind='bar')

plt.xticks(rotation=0)

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(data['text_weight'], bins=30, color='skyblue', edgecolor='black')
plt.title('Histogram dari Nilai Bobot dalam Kolom "text_weight"')
plt.xlabel('Nilai Bobot')
plt.ylabel('Frekuensi')
plt.grid(True)
plt.show()

## Train and Test Split

In [ ]:
Train_X, Test_X, Train_Y, Test_Y = train_test_split(data['full_text'], data['text_weight_label'], test_size=0.3, shuffle=True)

In [ ]:
tokenizer = Tokenizer(num_words=5000, oov_token='x', filters='!"#$%&()*+,-./:;<=>@[\]^_`{|}~ ')

tokenizer.fit_on_texts(data['full_text'])

# Konversi teks menjadi sequence
data_sequences = tokenizer.texts_to_sequences(data['full_text'])

padded_train = pad_sequences(data_sequences)

# Menampilkan hasil sequence
print(padded_train)

# tokenizer.fit_on_texts(Train_X)
# tokenizer.fit_on_texts(Test_X)

# sekuens_train = tokenizer.texts_to_sequences(Train_X)
# sekuens_test = tokenizer.texts_to_sequences(Test_X)

# padded_train = pad_sequences(sekuens_train)
# padded_test = pad_sequences(sekuens_test)

In [ ]:
print(f'Jumlah data dalam train set: {len(Train_X)}')
print(f'Jumlah data dalam validation set: {len(Test_X)}')
print(f'Jumlah data dalam train set: {len(Train_Y)}')
print(f'Jumlah data dalam validation set: {len(Test_Y)}')

In [ ]:
print(Train_Y)

## Vectorization

In [ ]:
Encoder = LabelEncoder()

Train_Y = Encoder.fit_transform(Train_Y)
Test_Y = Encoder.fit_transform(Test_Y)

In [ ]:
print(Train_Y)

In [ ]:
Tfidf_vect = TfidfVectorizer(max_features=5000)
Tfidf_vect.fit(data['full_text'])

Train_X_Tfidf = Tfidf_vect.transform(Train_X)
Test_X_Tfidf = Tfidf_vect.transform(Test_X)
Train_X.head()

## Klasifikasi

In [ ]:
Naive = naive_bayes.MultinomialNB()
Naive.fit(Train_X_Tfidf, Train_Y)


## Evaluasi

In [ ]:
predictions_NB = Naive.predict(Test_X_Tfidf)

print(confusion_matrix(predictions_NB,Test_Y))
print ("Classification: ",classification_report(Test_Y,predictions_NB))

In [ ]:
print("Naive Bayes Accuracy Score -> ", accuracy_score(predictions_NB, Test_Y))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(Test_Y, predictions_NB)
cm = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels=['Positif', 'Netral', 'Negatif'])
cm.plot()